<a href="https://colab.research.google.com/github/elhamod/IS834_Fall_2026/blob/main/Session%2004%20-%20Pandas%20II%20-%20Cleaning%20and%20Preparation/04_Analytics_with_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS834 — Session 4

Work through this notebook in Colab. The lecture cells are filled in and ready to run; the
**Exercises** are yours to complete — write your code in the `# Your code here` cells and your
reasoning in the `*Provide your answer here*` cells.

Every dataset is read straight from the course repository, so there is nothing to upload or install.

*Adapted from Brock Tibert's IS834 Fall 2025 session 4. Edited for Fall 2026 by Mohannad Elhamod.*

# Learning Goals

Session 3 loaded an unfamiliar dataset and described what was in it. This session turns that
description into a decision: what is wrong with the file, and what are you going to do about it?

- Auditing a dataset for quality problems
- Feature engineering -- creating new columns from existing ones
- Removing and renaming columns
- Changing data types
- Replacing values, including bad values and outliers
- Handling missing data
- String transformations
- Categorical data, nominal and ordinal
- Duplicates, and why the obvious check misses them

Everything below assumes session 3's toolkit: `read_csv`, `.info()`, `.describe()`, `.unique()`,
`.value_counts()` and filtering with `df[df.column > value]`. Those are how we *find* the problems.
The new material is what happens next.

In [ ]:
# pandas is the only package this notebook needs, and Colab already has it installed.
import pandas as pd

# Every data file this notebook uses is read straight from the course repository.
# There is nothing to upload: DATA is the folder those files live in.
DATA = "https://raw.githubusercontent.com/elhamod/IS834_Fall_2026/main/data/"


# Warmup Exercises

Five questions on last session's material, to get the loading-and-looking reflexes back before we
start changing anything.

**Exercise 1.** Read the diamonds dataset into a DataFrame called `diamonds` and look at the first few rows.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 2.** How many rows and columns does the dataset have?

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 3.** Retrieve the schema information for the dataset.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 4.** Are there any missing values in the dataset?

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 5.** Return a random sample of 3 rows.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

# Step 0: Audit Before You Clean

Cleaning is not a checklist you run. It is a set of decisions, and you cannot defend a decision you
made before you knew what was broken -- so the first thing we do is take an inventory of the damage.

The survey below is the same file we explored last session, freshly loaded and untouched.

In [ ]:
# Reload last session's survey. Nothing here has been cleaned yet.
df = pd.read_csv(DATA + "sample-survey-data.csv")
df.head(3)

In [ ]:
# Numeric problems hide in the min and max, not in the mean.
df[['Age', 'Income']].describe().T[['count', 'mean', 'min', 'max']]

> An age of **148** and an income of **-68,000**. Neither is possible, and both are already baked into
> the means printed right above them. The mean is the number people quote; the min and max are the
> numbers that tell you whether the mean is worth quoting.

In [ ]:
# Text problems never reach describe() at all -- you have to list the labels yourself.
print(df.Region.unique())
print(df.Recommend.unique())

In [ ]:
# dropna=False keeps the blanks in the table instead of quietly dropping them.
print(df.Region.value_counts(dropna=False))
print(df.Recommend.value_counts(dropna=False))

> `no` and `No` are the same answer recorded two ways, so a report counting `No` says 4 when the
> honest answer is 5 — two categories have quietly become three. And one respondent left `Region`
> blank, which is only visible because `dropna=False` kept the row in the table.

In [ ]:
# How much is missing, and in which columns?
df.isna().sum()

In [ ]:
# "Duplicate" depends on what you compare: whole rows, or the id meant to identify one person.
print(df.duplicated().sum())
print(df.duplicated(subset='id').sum())

**The audit, in one list.** Four blank answers (`Age`, `Satisfaction`, `Recommend` and `Region`), an
impossible age of 148, a negative income, a lowercase `no`, and — going by the last cell — a
respondent recorded twice that a whole-row comparison cannot see.

Summing booleans works here because Python counts `True` as 1, which is the trick that turns `.isna()`
from a table of True/False into a count.

The rest of this session is the toolkit for working through that list, and we come back to each item as
we reach the tool that fixes it. The duplicate is the one we leave for the exercises at the end,
because it is less about a command than about deciding what a row is supposed to represent.

# Feature Engineering


## New Columns


In [ ]:
# Creating a column works exactly like adding a new key to a dictionary: name it, assign to it.
df['age2'] = df.Age * 2
df.head(3)


In [ ]:
# .insert() is the version that lets you choose WHERE the column goes: (position, name, values).
df.insert(2, 'age_in_months', df.Age * 12)
df.head(3)


In [ ]:
# del removes a column we no longer want -- that one was just a demonstration.
del df['age_in_months']
df.head(3)

In [ ]:
# Columns can be combined with each other -- income per year of age.
df['inc_age'] = df.Income / df.Age
df[['Age', 'Income', 'inc_age']].head(3)


> Look at row 2: the income is −68,000, so `inc_age` is negative too. **Bad data does not stay in its own column.** Every feature you build on top of a dirty value inherits the problem, which is why cleaning comes before feature engineering in any real workflow.


In [ ]:
# String methods work on a column through the .str accessor.
df['edu_len'] = df.Education.str.len()
df[['Education', 'edu_len']].head(3)


> `.str` is the bridge between "one piece of text" and "a whole column of text". Anything you can do to a Python string — `.upper()`, `.strip()`, `.replace()`, `.contains()` — has a `.str` version that does it to every row at once.


# Follow up Questions and Exercises — Part 1: new columns


In [ ]:
# We use the diamonds dataset from the warmup exercise at the start of class.
# Reading it again here means this cell works whether or not you ran that exercise.
diamonds = pd.read_csv(DATA + "diamonds.csv")
diamonds.head(3)


**Exercise 6.** Create a new column holding the purchase price, which is 80% of the `price` column.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 7.** Calculate the price per carat for each row and store it in a new column.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 8.** Summarise the new price-per-carat column, and pull out its median.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 9.** Create a new column holding the `cut` values in upper case.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

## Remove Columns


In [ ]:
# The fast approach for a single column -- remember dictionaries.
del diamonds['rownames']
diamonds.head(3)


In [ ]:
# The other way is .drop with the columns argument, which takes a list of column names.
diamonds = diamonds.drop(columns=['table', 'z'])
diamonds.head(3)


> `.drop()` returns a **new** DataFrame; it does not change `diamonds` unless you assign the result back, which is why the line starts with `diamonds = `. Forgetting that assignment — and then wondering why the column is still there — is one of the most common pandas mistakes.


## Rename columns


In [ ]:
# Back to our small dataset.
df.head(3)


In [ ]:
# .rename with columns= and a dictionary of {old_name: new_name}.
df = df.rename(columns={'Age': 'age'})
df.head(3)


In [ ]:
# The dictionary can hold as many renames as you like.
df = df.rename(columns={'Income': 'inc', 'Region': 'reg'})
df.head(3)


> Renaming to short lowercase names without spaces is not just taste: it is what lets you write `df.age` instead of `df['Signup Date']`, and it is why almost every analyst's first line after loading a file is a column-name cleanup.


## Change Data Types


In [ ]:
# Check the schema before changing anything.
df.info()


In [ ]:
# .astype() converts a column to a different type.
df['inc_float'] = df.inc.astype(float)
df[['inc', 'inc_float']].head(3)


In [ ]:
# Converting a column that contains missing values to plain int fails. We catch the error so we can read it.
try:
    df['age_int'] = df.age.astype(int)
except Exception as error:
    print(type(error).__name__)
    print(error)


In [ ]:
# 'Int64' with a capital I is a nullable integer type -- it can hold whole numbers AND missing values.
df['age_int'] = df.age.astype('Int64')
df[['age', 'age_int']].head()


**Why the error, and why the capital I?** Plain `int` is a NumPy type with no concept of "missing" — every slot must hold a number, so a single blank age makes the conversion impossible. `'Int64'` is pandas' own nullable integer type, which reserves a slot for `<NA>`. The lowercase/uppercase difference is genuinely load-bearing here, and it is exactly the kind of question worth handing to an AI assistant when the error message alone does not click.


> Dates have a conversion of their own, `pd.to_datetime`, which we come back to further down. But the error above is a good excuse to talk about replacing values first.

## Replace Values


In [ ]:
# The starting point for this section.
df.head()


In [ ]:
# Replace values inside one column. The shape is {"column-name": {"old-value": "new-value"}}.
df = df.replace({"Education": {"Master": "Masters~~~"}})
df.head(3)


In [ ]:
# For a quick single change, work on the column directly with .replace(old, new).
df['Recommend'] = df['Recommend'].replace('Yes', 'Heck yeah')
df.Recommend.value_counts(dropna=False)


In [ ]:
# Now the same tool on a real defect: undo the joke and fold the lowercase "no" into "No".
df['Recommend'] = df['Recommend'].replace({'Heck yeah': 'Yes', 'no': 'No'})
df.Recommend.value_counts(dropna=False)


**This is the point of the section.** Before that line the column held four distinct answers — `Yes`, `No`, `no` and a blank — so counting "No" gave 4 when the honest answer was 5. Two categories had quietly become three. Standardising the spelling *before* you count is not tidying — it is the difference between a right answer and a wrong one.


In [ ]:
# Same reasoning for the impossible age of 148 -- flag it as missing rather than keep it.
df.loc[df.age > 120, 'age'] = pd.NA
df.age.describe()


In [ ]:
# And the negative income, which cannot be right either.
df.loc[df.inc < 0, 'inc'] = pd.NA
df.inc.describe()


> The mean age drops from 46.5 to 36.3 once that single 148 is removed, and mean income rises from about \$52,300 to \$63,200 once the negative value goes. **One bad row out of twelve moved the headline number by 10 years.** That is why the min/max line of `describe()` is worth reading before anything else.


In [ ]:
# When we have missing values we can use fillna. First, see where they are.
df.isna().sum()


In [ ]:
# Filling missing ages with -1 -- a common "placeholder" pattern.
df['age_clean'] = df.age.fillna(-1)
print(df.age_clean.tolist())
print('mean of age_clean:', df.age_clean.mean())


**And there is the trap.** −1 is a perfectly good number as far as pandas is concerned, so it goes straight into the average and drags it down. A placeholder only works if every later calculation knows to exclude it — and nothing in the code says so.


In [ ]:
# A more defensible fill: substitute the mean of the values we do have.
replacement_value = df.age.mean()
df['age_clean2'] = df.age.fillna(replacement_value)
print('replacement value:', round(replacement_value, 1))
print(df.age_clean2.round(1).tolist())


**Question 1.** The cell above filled the missing ages with the column mean instead of with -1. What does that buy you, what does it cost, and when would you not do it?

**Answer**

*Provide your answer here*

# Follow up Questions and Exercises — Part 2: removing, renaming and replacing


In [ ]:
# Setup - we use the diamonds dataframe from earlier in class.
diamonds.head(3)


**Exercise 10.** Remove the `depth` column from `diamonds`.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 11.** Rename the column `price` to `retail_price`.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 12.** Replace the `cut` value `Good` with `meh`.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

## Dates

`Signup Date` is still text, which is the loose end left at the end of the type-conversion section
above. Text sorts alphabetically, will not subtract, and cannot tell you a weekday.

In [ ]:
# Dates arrive as text. pd.to_datetime converts them into real dates.
df['signup_date'] = pd.to_datetime(df['Signup Date'])
df[['Signup Date', 'signup_date']].dtypes


In [ ]:
# Why bother? Because a real datetime can answer date questions for you, through the .dt accessor.
df['dow_str'] = df.signup_date.dt.day_name()
df['month'] = df.signup_date.dt.month_name()
df['year'] = df.signup_date.dt.year
df[['signup_date', 'dow_str', 'month', 'year']].head(3)


> `.dt` is to dates what `.str` is to text: the bridge between one value and a whole column of them. Weekday, month name, quarter, "days since" — none of that is available while the column is still text.


## Categorical Data

This is how you tell pandas which columns are labels, not free text or numbers. Doing so saves memory, improves speed, keeps data clean, and can help with your downstream analytical tasks!  

It's not __always__ a step we have to do, but as a rule, properly setting each column's data type is a best practice.


![](https://media.geeksforgeeks.org/wp-content/uploads/20230918125039/2.png)

![](https://media.geeksforgeeks.org/wp-content/uploads/20230918125108/3.png)


In [ ]:
# Re-load the small dataset into a fresh variable so the renames and fixes above do not confuse things.
df2 = pd.read_csv(DATA + "sample-survey-data.csv")
df2.sample(3)


In [ ]:
# 1. categorical: "Region"
df2["region_cat"] = df2["Region"].astype("category")
df2.region_cat.dtype


**Question 2.** Is `Region` a nominal or an ordinal category?

**Answer**

*Provide your answer here*

In [ ]:
# Now Satisfaction. First, what values are actually in there?
df2.Satisfaction.unique()


In [ ]:
# pd.Categorical lets us state the categories AND their order.
df2["satis_cat"] = pd.Categorical(
    df2["Satisfaction"],
    categories=[1, 2, 3, 4, 5],   # these could easily be strings, e.g. ['Bronze', 'Silver', 'Gold']
    ordered=True                  # this is the important part
)
df2.satis_cat.dtype


In [ ]:
# Why bother? Because the values now stay in a meaningful order instead of an accidental one.
df2.satis_cat.value_counts(sort=False)


**Read the two tables side by side.** `value_counts()` on the raw column sorts by frequency, so the ratings come out in whatever order happens to be most common — useless for a chart. With `sort=False` on an ordered category, 1 through 5 appear in rating order, and **rating 1 shows up with a count of zero**. A plain `value_counts()` cannot show you a category that nobody chose, because it only knows about values that exist. Telling pandas the full set of valid categories is what makes the gap visible.


# Follow up Questions and Exercises — Part 3: categorical and date columns


**Exercise 13.** Turn `Education` into an ordinal categorical column, then count the rows per level.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 14.** Fix the datatype of `Signup Date` and extract the year into its own column.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 15.** Divide income by 12 and round the result to a whole number.

> TIP: `help(pd.Series.round)` tells you what the argument does.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

## Numeric -> Categorical


In [ ]:
# pd.cut slices a numeric column into bands. Here: three equal-width bands of income.
df2['income_bands'] = pd.cut(df2.Income, 3)
df2.income_bands.value_counts(sort=False)


**How to read those labels.** `(-15000.0, 38000.0]` means "greater than −15,000, up to and including 38,000" — a round bracket excludes the endpoint, a square bracket includes it.

**And look at the counts: 1, 1, 10.** `pd.cut` with a number splits the *range* into equal-width slices, and the range here runs from −68,000 to 91,000 because of the one impossible negative income. A single bad value has made two of the three bands almost empty and dumped 10 of 12 respondents into the third. The bands are useless.


**Exercise 16.** The bands above are useless because one impossible income stretched the range. Mark that value missing, band the column again, and give the three bands readable labels.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

# Mini Practice Problem


**Exercise 17.** Your firm has just begun to formally track and collect customer complaint data. The tracking system and process are new, so the data is not yet perfect.

You are an analyst supporting the customer service manager. The manager wants your help to understand the main sources of complaints and is looking for guidance on where to focus first.

Based on your analysis of the complaint data, what is your recommendation for where the manager should spend their time?

In [ ]:
# Starter code: a list of dictionaries, which pd.DataFrame turns straight into a table.
data = [
    {"complaint_id": "C001", "category": "Card",    "description": "Charged twice",               "date_submitted": "2025-07-01", "tenure_months": 12},
    {"complaint_id": "C002", "category": "Cards",   "description": "Card not received",           "date_submitted": "2025-07-02", "tenure_months": 24},
    {"complaint_id": "C003", "category": "CRAD",    "description": "Card declined at POS",        "date_submitted": "2025-07-02", "tenure_months": None},
    {"complaint_id": "C004", "category": "ATM",     "description": "Cash not dispensed",          "date_submitted": "2025-07-03", "tenure_months": 6},
    {"complaint_id": "C005", "category": "atm",     "description": "Machine out of service",      "date_submitted": "2025-07-03", "tenure_months": 8},
    {"complaint_id": "C006", "category": "Loans",   "description": "Late fee dispute",            "date_submitted": "2025-07-04", "tenure_months": None},
    {"complaint_id": "C007", "category": "Loan",    "description": "Payment not applied",         "date_submitted": "2025-07-05", "tenure_months": 18},
    {"complaint_id": "C008", "category": "checking","description": "Unexpected monthly fee",      "date_submitted": "2025-07-06", "tenure_months": 3},
    {"complaint_id": "C009", "category": "Checking","description": "Overdraft fee issue",         "date_submitted": "2025-07-06", "tenure_months": None},
    {"complaint_id": "C010", "category": "Mortg.",  "description": "Escrow analysis confusion",   "date_submitted": "2025-07-07", "tenure_months": 30},
    {"complaint_id": "C011", "category": "Mortgage","description": "Rate adjustment unclear",     "date_submitted": "2025-07-07", "tenure_months": 36},
    {"complaint_id": "C012", "category": "Card",    "description": "Lost card replacement delay", "date_submitted": "2025-07-08", "tenure_months": 5},
    {"complaint_id": "C013", "category": "ATM",     "description": "Deposit not posted",          "date_submitted": "2025-07-09", "tenure_months": None},
    {"complaint_id": "C014", "category": "Cards",   "description": "Incorrect card limit",        "date_submitted": "2025-07-10", "tenure_months": 20},
    {"complaint_id": "C015", "category": "Loan",    "description": "Prepayment penalty confusion","date_submitted": "2025-07-10", "tenure_months": 15},
]

complaints = pd.DataFrame(data)
complaints


In [ ]:
# Your code here -- step 1: look before you leap (shape, types, missing values)

In [ ]:
# Your code here -- step 2: count complaints by category, then clean the labels and count again

In [ ]:
# Your code here -- step 3: turn the counts into shares, and check how usable tenure_months is

**Answer**

*Provide your answer here*

# Follow up Questions and Exercises — Part 4: auditing and duplicates

**Question 3.** `describe()` on the survey data ignored `userid`, `Education`, `Region`, `Recommend` and `comment`. What could go wrong if that summary were the only quality check you ran before reporting?


**Answer**

*Provide your answer here*

**Question 4.** We filled missing ages twice — once with −1 and once with the column mean. Both are defensible in some situation and indefensible in others. When is each one appropriate?


**Answer**

*Provide your answer here*

**Exercise 18.** The audit at the top found that `duplicated()` and `duplicated(subset='id')` disagree. Work out *which* respondent is duplicated, and why comparing whole rows misses it.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 19.** A colleague suggests starting the analysis with `survey.dropna()`. Show what that costs on this dataset, and decide whether it is a good idea.


In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 20.** The `userid` column looks like it should uniquely identify a person. Check whether it does, and work out what would happen if you de-duplicated on it.


In [ ]:
# Your code here

**Answer**

*Provide your answer here*